# CIC-MalMem-2022 Revalidation — 02 Classical Augmentation

**Team B — Generative Augmentation and Quantum Crossover**  
**Author:** Marco Matya  
**Protocol:** Revalidation v2

This notebook implements **SMOTE**, **Borderline-SMOTE**, and **ADASYN**
without changing the locked dataset split created by Notebook 01.

## Two distinct research questions

### A. Matched QGAN crossover

Generate exactly **2,000 additional Trojan observations** with each
classical augmentation method. The synthetic volume and target class are
therefore matched to the QGAN crossover protocol.

### B. Multiclass augmentation-budget selection

For each classical method, augment Ransomware, Spyware, and Trojan by 25%,
50%, or 100% of their deficit relative to Benign. Select the budget using
**validation Macro-F1 only**. The test partition is not evaluated here.

## 0. Environment

Recommended package version for this protocol:

```python
%pip install "imbalanced-learn==0.14.2"
```


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import shutil
import sys
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.neighbors import NearestNeighbors

try:
    import imblearn
    from imblearn.over_sampling import ADASYN, BorderlineSMOTE, SMOTE
except ImportError as error:
    raise ImportError(
        "Install the required package with: "
        "%pip install 'imbalanced-learn==0.14.2', restart the kernel, "
        "and rerun from the beginning."
    ) from error

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("default")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)


## 1. Configuration


In [ ]:
def resolve_project_root(project_name: str = "Marco_Revalidation_v2") -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        nested = candidate / project_name
        if nested.exists():
            return nested.resolve()
    return (cwd / project_name).resolve()


PROJECT_ROOT = resolve_project_root()
CONTRACT_DIR = PROJECT_ROOT / "01_data_contract"
OUTPUT_DIR = PROJECT_ROOT / "02_classical_augmentation" / "outputs"

CLEAN_DATA_PATH = CONTRACT_DIR / "malmem2022_clean_locked.csv.gz"
SPLIT_MANIFEST_PATH = CONTRACT_DIR / "split_manifest.csv"
PREPROCESSOR_PATH = CONTRACT_DIR / "preprocessing.joblib"
PREPROCESSED_SPLITS_PATH = CONTRACT_DIR / "locked_preprocessed_splits.npz"
DATASET_MANIFEST_PATH = CONTRACT_DIR / "dataset_manifest.json"

EXPECTED_DATASET_SHA256 = (
    "cc7a637a174ffe797e0af0375bce3c09561f0dc8b8115c0a6292718034f5012a"
)
EXPECTED_PARTITION_SIZES = {
    "train": 40_642,
    "validation": 8_710,
    "test": 8_710,
}
EXPECTED_RETAINED_FEATURES = 52

TARGET = "Label"
CLASS_ORDER = ["Benign", "Ransomware", "Spyware", "Trojan"]
SEED = 42

METHODS = ["SMOTE", "BorderlineSMOTE", "ADASYN"]
MATCHED_TARGET_CLASS = "Trojan"
MATCHED_SYNTHETIC_COUNT = 2_000
MULTICLASS_BUDGETS = [0.25, 0.50, 1.00]

K_NEIGHBORS = 5
M_NEIGHBORS = 10
MAX_GENERATION_ATTEMPTS = 6
VALIDATION_RF_TREES = 150

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "matched").mkdir(exist_ok=True)
(OUTPUT_DIR / "multiclass_budget").mkdir(exist_ok=True)
(OUTPUT_DIR / "selected").mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Contract directory:", CONTRACT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


## 2. Load and verify the locked contract


In [ ]:
required_paths = [
    CLEAN_DATA_PATH,
    SPLIT_MANIFEST_PATH,
    PREPROCESSOR_PATH,
    PREPROCESSED_SPLITS_PATH,
    DATASET_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing Notebook 01 artifacts:\n- " + "\n- ".join(missing_paths)
    )

dataset_manifest = json.loads(DATASET_MANIFEST_PATH.read_text(encoding="utf-8"))
preprocessing_bundle = joblib.load(PREPROCESSOR_PATH)
clean = pd.read_csv(CLEAN_DATA_PATH, low_memory=False)
split_manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
locked_npz = np.load(PREPROCESSED_SPLITS_PATH)

assert dataset_manifest["source_sha256"] == EXPECTED_DATASET_SHA256
assert preprocessing_bundle["dataset_sha256"] == EXPECTED_DATASET_SHA256
assert len(clean) == 58_062
assert clean["row_id"].is_unique
assert clean[TARGET].value_counts().reindex(CLASS_ORDER).astype(int).to_dict() == {
    "Benign": 29_231,
    "Ransomware": 9_529,
    "Spyware": 9_815,
    "Trojan": 9_487,
}

observed_partition_sizes = clean["partition"].value_counts().to_dict()
assert observed_partition_sizes == EXPECTED_PARTITION_SIZES
assert len(preprocessing_bundle["retained_features"]) == EXPECTED_RETAINED_FEATURES

split_check = clean[["row_id", "model_group_id", TARGET, "partition"]].merge(
    split_manifest[["row_id", "model_group_id", TARGET, "partition"]],
    on="row_id",
    suffixes=("_clean", "_manifest"),
    validate="one_to_one",
)
for column in ["model_group_id", TARGET, "partition"]:
    assert split_check[f"{column}_clean"].equals(split_check[f"{column}_manifest"])

cross_partition_groups = (
    clean.groupby("model_group_id")["partition"].nunique().gt(1).sum()
)
assert int(cross_partition_groups) == int(
    dataset_manifest["cross_partition_model_group_count"]
)

print("Locked data contract: VERIFIED")
print("Partition sizes:", observed_partition_sizes)
print("Retained features:", len(preprocessing_bundle["retained_features"]))
print("Feature-equivalent groups crossing partitions:", int(cross_partition_groups))


## 3. Reconstruct the saved train-only representation



In [ ]:
feature_columns = preprocessing_bundle["feature_columns"]
retained_features = preprocessing_bundle["retained_features"]
retained_mask = np.asarray(preprocessing_bundle["retained_mask"], dtype=bool)
imputer = preprocessing_bundle["imputer"]
scaler = preprocessing_bundle["scaler"]

partition_frames = {
    name: clean.loc[clean["partition"].eq(name)].reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

def transform_locked(frame: pd.DataFrame) -> np.ndarray:
    imputed = imputer.transform(frame[feature_columns])
    return scaler.transform(imputed[:, retained_mask])

X_train = transform_locked(partition_frames["train"])
X_validation = transform_locked(partition_frames["validation"])
X_test = transform_locked(partition_frames["test"])

y_train = partition_frames["train"][TARGET].to_numpy(dtype=str)
y_validation = partition_frames["validation"][TARGET].to_numpy(dtype=str)
y_test = partition_frames["test"][TARGET].to_numpy(dtype=str)

assert np.allclose(X_train, locked_npz["X_train"], rtol=1e-6, atol=1e-6)
assert np.allclose(X_validation, locked_npz["X_validation"], rtol=1e-6, atol=1e-6)
assert np.allclose(X_test, locked_npz["X_test"], rtol=1e-6, atol=1e-6)
assert np.array_equal(y_train, locked_npz["y_train"])
assert np.array_equal(y_validation, locked_npz["y_validation"])
assert np.array_equal(y_test, locked_npz["y_test"])

train_counts = pd.Series(y_train).value_counts().reindex(CLASS_ORDER).astype(int)
display(train_counts.rename("train_count").to_frame())
print("Locked matrix equality checks: PASSED")


## 4. Training-domain profile and reconstruction functions


In [ ]:
X_train_imputed_full = imputer.transform(
    partition_frames["train"][feature_columns]
)
train_min = np.min(X_train_imputed_full, axis=0)
train_max = np.max(X_train_imputed_full, axis=0)
integer_like_mask = np.max(
    np.abs(X_train_imputed_full - np.rint(X_train_imputed_full)), axis=0
) <= 1e-9

def reconstruct_original_space(X_scaled: np.ndarray):
    retained_raw = scaler.inverse_transform(X_scaled)
    full_raw = np.tile(imputer.statistics_, (len(X_scaled), 1)).astype(float)
    full_raw[:, retained_mask] = retained_raw

    constrained = np.clip(full_raw, train_min, train_max)
    constrained[:, integer_like_mask] = np.rint(
        constrained[:, integer_like_mask]
    )
    return full_raw, constrained

domain_profile = pd.DataFrame({
    "feature": feature_columns,
    "retained": retained_mask,
    "train_min": train_min,
    "train_max": train_max,
    "training_integer_like": integer_like_mask,
    "imputer_statistic": imputer.statistics_,
})
domain_profile.to_csv(OUTPUT_DIR / "training_domain_profile.csv", index=False)
display(domain_profile.head(12))


## 5. Exact-count generation utilities


In [6]:
def slugify_method(method: str) -> str:
    return method.lower().replace("-", "_")


def make_sampler(method: str, target_class: str, requested_final_count: int, seed: int):
    strategy = {target_class: int(requested_final_count)}
    if method == "SMOTE":
        return SMOTE(
            sampling_strategy=strategy,
            random_state=seed,
            k_neighbors=K_NEIGHBORS,
        )
    if method == "BorderlineSMOTE":
        return BorderlineSMOTE(
            sampling_strategy=strategy,
            random_state=seed,
            k_neighbors=K_NEIGHBORS,
            m_neighbors=M_NEIGHBORS,
            kind="borderline-1",
        )
    if method == "ADASYN":
        return ADASYN(
            sampling_strategy=strategy,
            random_state=seed,
            n_neighbors=K_NEIGHBORS,
        )
    raise ValueError(f"Unknown method: {method}")


def _extract_appended_synthetic(X_original, y_original, X_resampled, y_resampled):
    n_original = len(X_original)
    if len(X_resampled) < n_original:
        raise RuntimeError("Oversampler returned fewer rows than the original training set.")
    if not np.allclose(X_resampled[:n_original], X_original):
        raise RuntimeError(
            "Oversampler did not preserve original observations as the leading block; "
            "synthetic-row extraction would be ambiguous."
        )
    if not np.array_equal(np.asarray(y_resampled[:n_original]), np.asarray(y_original)):
        raise RuntimeError("Oversampler changed the order of original labels.")
    return (
        np.asarray(X_resampled[n_original:], dtype=float),
        np.asarray(y_resampled[n_original:], dtype=str),
    )


def generate_exact_for_class(
    method: str,
    target_class: str,
    n_to_generate: int,
    base_seed: int = SEED,
):
    if n_to_generate <= 0:
        return np.empty((0, X_train.shape[1]), dtype=float), []

    original_count = int(np.sum(y_train == target_class))
    candidate_batches = []
    attempt_log = []

    for attempt in range(MAX_GENERATION_ATTEMPTS):
        attempt_seed = base_seed + attempt
        # ADASYN can undershoot because local allocations are rounded.
        oversupply = int(np.ceil(n_to_generate * (1.10 if method == "ADASYN" else 1.00)))
        requested_final = original_count + oversupply
        sampler = make_sampler(method, target_class, requested_final, attempt_seed)

        start = time.perf_counter()
        X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
        elapsed = time.perf_counter() - start
        X_new, y_new = _extract_appended_synthetic(
            X_train, y_train, X_resampled, y_resampled
        )
        X_class = X_new[y_new == target_class]
        candidate_batches.append(X_class)
        attempt_log.append({
            "method": method,
            "target_class": target_class,
            "attempt": attempt + 1,
            "seed": attempt_seed,
            "requested_synthetic": oversupply,
            "returned_synthetic_for_class": len(X_class),
            "seconds": elapsed,
        })

        combined = np.vstack(candidate_batches)
        rounded = np.round(combined, decimals=12)
        _, unique_indices = np.unique(rounded, axis=0, return_index=True)
        unique_candidates = combined[np.sort(unique_indices)]
        if len(unique_candidates) >= n_to_generate:
            rng = np.random.default_rng(base_seed)
            selected = rng.choice(
                len(unique_candidates), size=n_to_generate, replace=False
            )
            return unique_candidates[selected], attempt_log

    raise RuntimeError(
        f"{method} produced fewer than {n_to_generate} unique candidates "
        f"for {target_class} after {MAX_GENERATION_ATTEMPTS} attempts. "
        "Do not fill the gap with copied rows or another algorithm."
    )


def generate_exact_multiclass(method: str, additions: dict, base_seed: int = SEED):
    matrices = []
    labels = []
    logs = []
    for class_offset, class_name in enumerate(CLASS_ORDER):
        n_to_generate = int(additions.get(class_name, 0))
        if n_to_generate <= 0:
            continue
        X_class, class_logs = generate_exact_for_class(
            method,
            class_name,
            n_to_generate,
            base_seed=base_seed + 100 * class_offset,
        )
        matrices.append(X_class)
        labels.extend([class_name] * len(X_class))
        logs.extend(class_logs)

    if not matrices:
        return np.empty((0, X_train.shape[1])), np.empty(0, dtype=str), logs
    X_synthetic = np.vstack(matrices)
    y_synthetic = np.asarray(labels, dtype=str)
    assert len(X_synthetic) == sum(int(v) for v in additions.values())
    assert np.isfinite(X_synthetic).all()
    return X_synthetic, y_synthetic, logs


## 6. Quality-control and persistence utilities


In [7]:
nearest_real_model = NearestNeighbors(n_neighbors=1, algorithm="auto")
nearest_real_model.fit(X_train)

def synthetic_quality_record(
    experiment: str,
    method: str,
    budget_label: str,
    X_synthetic: np.ndarray,
    y_synthetic: np.ndarray,
) -> dict:
    rounded = np.round(X_synthetic, decimals=12)
    unique_rows = len(np.unique(rounded, axis=0))
    distances, _ = nearest_real_model.kneighbors(X_synthetic, return_distance=True)
    nearest_distances = distances[:, 0]
    return {
        "experiment": experiment,
        "method": method,
        "budget": budget_label,
        "synthetic_rows": int(len(X_synthetic)),
        "synthetic_unique_rows": int(unique_rows),
        "synthetic_unique_fraction": float(unique_rows / max(len(X_synthetic), 1)),
        "exact_scaled_overlap_with_real": int(np.sum(nearest_distances <= 1e-12)),
        "nearest_real_distance_mean": float(np.mean(nearest_distances)),
        "nearest_real_distance_median": float(np.median(nearest_distances)),
        "nearest_real_distance_min": float(np.min(nearest_distances)),
        "finite": bool(np.isfinite(X_synthetic).all()),
    }


def save_synthetic_artifacts(
    directory: Path,
    prefix: str,
    method: str,
    X_synthetic: np.ndarray,
    y_synthetic: np.ndarray,
):
    directory.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        directory / f"{prefix}_synthetic_scaled.npz",
        X_synthetic=X_synthetic.astype(np.float32),
        y_synthetic=y_synthetic.astype(str),
        retained_features=np.asarray(retained_features, dtype=str),
        method=np.asarray([method], dtype=str),
    )

    raw_full, constrained_full = reconstruct_original_space(X_synthetic)
    raw_frame = pd.DataFrame(raw_full, columns=feature_columns)
    constrained_frame = pd.DataFrame(constrained_full, columns=feature_columns)
    for frame in [raw_frame, constrained_frame]:
        frame.insert(0, TARGET, y_synthetic)
        frame.insert(0, "source", "synthetic")
        frame.insert(0, "method", method)

    raw_frame.to_csv(
        directory / f"{prefix}_synthetic_raw_reconstructed.csv.gz",
        index=False,
        compression="gzip",
    )
    constrained_frame.to_csv(
        directory / f"{prefix}_synthetic_constrained.csv.gz",
        index=False,
        compression="gzip",
    )


## 7. Experiment A — matched +2,000 Trojan


In [ ]:
matched_additions = {MATCHED_TARGET_CLASS: MATCHED_SYNTHETIC_COUNT}
matched_datasets = {}
generation_logs = []
quality_records = []

for method_index, method in enumerate(METHODS):
    print(f"Generating matched dataset: {method}")
    X_syn, y_syn, logs = generate_exact_multiclass(
        method,
        matched_additions,
        base_seed=SEED + 1_000 * method_index,
    )
    assert Counter(y_syn) == Counter(matched_additions)

    matched_datasets[method] = (X_syn, y_syn)
    generation_logs.extend(logs)
    quality_records.append(
        synthetic_quality_record(
            "matched_trojan_2000", method, "+2000_Trojan", X_syn, y_syn
        )
    )
    slug = slugify_method(method)
    save_synthetic_artifacts(
        OUTPUT_DIR / "matched",
        f"matched_{slug}",
        method,
        X_syn,
        y_syn,
    )

matched_distribution_rows = []
for method, (X_syn, y_syn) in matched_datasets.items():
    augmented_counts = Counter(y_train)
    augmented_counts.update(y_syn)
    for class_name in CLASS_ORDER:
        matched_distribution_rows.append({
            "method": method,
            "class": class_name,
            "real_train": int(np.sum(y_train == class_name)),
            "synthetic": int(np.sum(y_syn == class_name)),
            "augmented_total": int(augmented_counts[class_name]),
        })

matched_distribution = pd.DataFrame(matched_distribution_rows)
display(matched_distribution)


## 8. Experiment B — multiclass augmentation budgets


In [ ]:
majority_count = int(train_counts.max())

def additions_for_budget(budget: float) -> dict:
    additions = {}
    for class_name in CLASS_ORDER:
        current = int(train_counts[class_name])
        if current >= majority_count:
            continue
        additions[class_name] = int(np.rint(budget * (majority_count - current)))
    return additions

budget_plan_rows = []
for budget in MULTICLASS_BUDGETS:
    additions = additions_for_budget(budget)
    for class_name in CLASS_ORDER:
        current = int(train_counts[class_name])
        added = int(additions.get(class_name, 0))
        budget_plan_rows.append({
            "budget": budget,
            "class": class_name,
            "real_train": current,
            "requested_synthetic": added,
            "target_total": current + added,
        })

budget_plan = pd.DataFrame(budget_plan_rows)
display(budget_plan)


In [ ]:
multiclass_datasets = {}

for method_index, method in enumerate(METHODS):
    for budget_index, budget in enumerate(MULTICLASS_BUDGETS):
        budget_pct = int(round(100 * budget))
        key = (method, budget)
        additions = additions_for_budget(budget)
        print(f"Generating {method}, budget={budget_pct}%: {additions}")

        X_syn, y_syn, logs = generate_exact_multiclass(
            method,
            additions,
            base_seed=SEED + 10_000 * method_index + 1_000 * budget_index,
        )
        assert Counter(y_syn) == Counter(additions)
        multiclass_datasets[key] = (X_syn, y_syn)
        generation_logs.extend(logs)
        quality_records.append(
            synthetic_quality_record(
                "multiclass_budget",
                method,
                f"{budget_pct}%",
                X_syn,
                y_syn,
            )
        )

        slug = slugify_method(method)
        prefix = f"{slug}_budget_{budget_pct:03d}"
        save_synthetic_artifacts(
            OUTPUT_DIR / "multiclass_budget",
            prefix,
            method,
            X_syn,
            y_syn,
        )

generation_log = pd.DataFrame(generation_logs)
quality_table = pd.DataFrame(quality_records)
generation_log.to_csv(OUTPUT_DIR / "generation_attempt_log.csv", index=False)
quality_table.to_csv(OUTPUT_DIR / "synthetic_quality_checks.csv", index=False)
display(quality_table)


## 9. Validation-only budget selection


In [ ]:
def evaluate_validation_condition(
    condition: str,
    method: str,
    budget,
    X_synthetic: np.ndarray,
    y_synthetic: np.ndarray,
) -> dict:
    X_fit = np.vstack([X_train, X_synthetic])
    y_fit = np.concatenate([y_train, y_synthetic])

    model = RandomForestClassifier(
        n_estimators=VALIDATION_RF_TREES,
        random_state=SEED,
        n_jobs=-1,
        class_weight=None,
    )
    start = time.perf_counter()
    model.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - start
    predictions = model.predict(X_validation)

    per_class_f1 = f1_score(
        y_validation,
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0,
    )
    record = {
        "condition": condition,
        "method": method,
        "budget": budget,
        "train_rows_total": len(X_fit),
        "synthetic_rows": len(X_synthetic),
        "validation_macro_f1": f1_score(
            y_validation, predictions, average="macro", zero_division=0
        ),
        "validation_weighted_f1": f1_score(
            y_validation, predictions, average="weighted", zero_division=0
        ),
        "validation_mcc": matthews_corrcoef(y_validation, predictions),
        "validation_balanced_accuracy": balanced_accuracy_score(
            y_validation, predictions
        ),
        "fit_seconds": fit_seconds,
    }
    for class_name, value in zip(CLASS_ORDER, per_class_f1):
        record[f"validation_f1_{class_name.lower()}"] = float(value)
    return record


validation_records = [
    evaluate_validation_condition(
        "Original", "Original", 0.0,
        np.empty((0, X_train.shape[1])),
        np.empty(0, dtype=str),
    )
]

for (method, budget), (X_syn, y_syn) in multiclass_datasets.items():
    validation_records.append(
        evaluate_validation_condition(
            f"{method}_{int(round(100 * budget))}%",
            method,
            budget,
            X_syn,
            y_syn,
        )
    )

validation_results = pd.DataFrame(validation_records)
validation_results.to_csv(
    OUTPUT_DIR / "validation_budget_selection_results.csv", index=False
)
display(
    validation_results.sort_values(
        "validation_macro_f1", ascending=False
    ).reset_index(drop=True)
)


## 10. Select one multiclass budget per method


In [ ]:
selected_rows = []
selected_datasets = {}

for method in METHODS:
    candidates = validation_results[
        validation_results["method"].eq(method)
    ].copy()
    candidates = candidates.sort_values(
        ["validation_macro_f1", "budget"],
        ascending=[False, True],
    )
    winner = candidates.iloc[0]
    selected_rows.append(winner.to_dict())

    selected_budget = float(winner["budget"])
    X_syn, y_syn = multiclass_datasets[(method, selected_budget)]
    selected_datasets[method] = (selected_budget, X_syn, y_syn)

    slug = slugify_method(method)
    budget_pct = int(round(100 * selected_budget))
    source_prefix = f"{slug}_budget_{budget_pct:03d}"
    for suffix in [
        "synthetic_scaled.npz",
        "synthetic_raw_reconstructed.csv.gz",
        "synthetic_constrained.csv.gz",
    ]:
        source = OUTPUT_DIR / "multiclass_budget" / f"{source_prefix}_{suffix}"
        destination = OUTPUT_DIR / "selected" / f"selected_{slug}_{suffix}"
        shutil.copy2(source, destination)

selected_budgets = pd.DataFrame(selected_rows)
selected_budgets.to_csv(OUTPUT_DIR / "selected_budgets.csv", index=False)
display(
    selected_budgets[[
        "method", "budget", "synthetic_rows",
        "validation_macro_f1", "validation_mcc",
        "validation_balanced_accuracy",
    ]]
)

print("TEST LABELS ACCESSED FOR SCORING: False")


## 11. Figures



In [ ]:
plt.figure(figsize=(10, 5.5))
sns.barplot(
    data=matched_distribution,
    x="class",
    y="augmented_total",
    hue="method",
    order=CLASS_ORDER,
)
plt.title("Matched experiment: real train + exactly 2,000 synthetic Trojan")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "01_matched_class_distributions.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

plot_results = validation_results[validation_results["method"].ne("Original")].copy()
plot_results["budget_percent"] = 100 * plot_results["budget"].astype(float)
plt.figure(figsize=(8.5, 5.2))
sns.lineplot(
    data=plot_results,
    x="budget_percent",
    y="validation_macro_f1",
    hue="method",
    marker="o",
)
original_macro = float(
    validation_results.loc[
        validation_results["method"].eq("Original"), "validation_macro_f1"
    ].iloc[0]
)
plt.axhline(
    original_macro,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Original baseline",
)
plt.title("Validation-only augmentation-budget selection")
plt.xlabel("Fraction of class deficit closed (%)")
plt.ylabel("Validation Macro-F1")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "02_validation_budget_selection.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

pca = PCA(n_components=2, random_state=SEED)
pca.fit(X_train)
rng = np.random.default_rng(SEED)
real_indices = rng.choice(len(X_train), size=min(2_000, len(X_train)), replace=False)
pca_frames = [pd.DataFrame({
    "PC1": pca.transform(X_train[real_indices])[:, 0],
    "PC2": pca.transform(X_train[real_indices])[:, 1],
    "source": "Real train",
})]
for method, (X_syn, _) in matched_datasets.items():
    embedding = pca.transform(X_syn)
    pca_frames.append(pd.DataFrame({
        "PC1": embedding[:, 0],
        "PC2": embedding[:, 1],
        "source": method,
    }))
pca_plot = pd.concat(pca_frames, ignore_index=True)
plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="source",
    alpha=0.35,
    s=18,
    linewidth=0,
)
plt.title("Matched Trojan augmentation in train-fitted PCA space")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "03_matched_pca_preview.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## 12. Save protocol manifest and hand-off package


In [ ]:
created_at_utc = datetime.now(timezone.utc).isoformat()

matched_distribution.to_csv(
    OUTPUT_DIR / "matched_class_distributions.csv", index=False
)
budget_plan.to_csv(OUTPUT_DIR / "multiclass_budget_plan.csv", index=False)

selected_manifest = {
    row["method"]: {
        "budget": float(row["budget"]),
        "synthetic_rows": int(row["synthetic_rows"]),
        "validation_macro_f1": float(row["validation_macro_f1"]),
        "validation_mcc": float(row["validation_mcc"]),
        "validation_balanced_accuracy": float(
            row["validation_balanced_accuracy"]
        ),
    }
    for _, row in selected_budgets.iterrows()
}

augmentation_manifest = {
    "protocol_version": "revalidation_v2_classical_augmentation_v1",
    "created_at_utc": created_at_utc,
    "input_dataset_sha256": dataset_manifest["source_sha256"],
    "input_contract_protocol": dataset_manifest["protocol_version"],
    "split_seed": dataset_manifest["split"]["seed"],
    "partition_sizes": observed_partition_sizes,
    "retained_feature_count": len(retained_features),
    "cross_partition_model_group_count": int(cross_partition_groups),
    "cross_partition_disclosure": (
        "Twelve feature-equivalent groups cross the locked comparability split. "
        "No split was recreated here; a group-aware sensitivity analysis is "
        "required before a final publication claim."
    ),
    "methods": METHODS,
    "sampler_parameters": {
        "k_neighbors": K_NEIGHBORS,
        "m_neighbors_borderline": M_NEIGHBORS,
        "borderline_kind": "borderline-1",
        "base_seed": SEED,
        "max_exact_count_attempts": MAX_GENERATION_ATTEMPTS,
    },
    "matched_experiment": {
        "target_class": MATCHED_TARGET_CLASS,
        "synthetic_count_per_method": MATCHED_SYNTHETIC_COUNT,
    },
    "multiclass_budgets": MULTICLASS_BUDGETS,
    "budget_selection_model": {
        "estimator": "RandomForestClassifier",
        "n_estimators": VALIDATION_RF_TREES,
        "random_state": SEED,
        "selection_metric": "validation_macro_f1",
        "tie_breaker": "smaller_budget",
    },
    "selected_multiclass_budgets": selected_manifest,
    "test_scored": False,
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "imbalanced_learn": imblearn.__version__,
        "joblib": joblib.__version__,
    },
}
(OUTPUT_DIR / "augmentation_manifest.json").write_text(
    json.dumps(augmentation_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

selected_lines = "\n".join(
    f"- {method}: {100 * values['budget']:.0f}% budget; "
    f"validation Macro-F1={values['validation_macro_f1']:.4f}"
    for method, values in selected_manifest.items()
)
readme = f'''# Revalidation v2 — classical augmentation

Input contract SHA-256: `{dataset_manifest["source_sha256"]}`

## Matched branch

SMOTE, Borderline-SMOTE, and ADASYN each generated exactly 2,000 Trojan
samples using locked training observations only.

## Validation-selected multiclass branch

{selected_lines}

'''
(OUTPUT_DIR / "README.md").write_text(readme, encoding="utf-8")

archive_path = shutil.make_archive(
    str(PROJECT_ROOT / "02_classical_augmentation" / "02_classical_augmentation_handoff"),
    "zip",
    root_dir=OUTPUT_DIR,
)
print("Saved augmentation artifacts to:", OUTPUT_DIR.resolve())
print("Hand-off archive:", archive_path)


## 13. Final acceptance checks

In [ ]:
reloaded_manifest = json.loads(
    (OUTPUT_DIR / "augmentation_manifest.json").read_text(encoding="utf-8")
)
reloaded_selected = pd.read_csv(OUTPUT_DIR / "selected_budgets.csv")
reloaded_quality = pd.read_csv(OUTPUT_DIR / "synthetic_quality_checks.csv")

assert reloaded_manifest["input_dataset_sha256"] == EXPECTED_DATASET_SHA256
assert reloaded_manifest["test_scored"] is False
assert set(reloaded_selected["method"]) == set(METHODS)
assert reloaded_quality["finite"].astype(bool).all()
assert (reloaded_quality["synthetic_unique_fraction"] > 0.99).all()

for method in METHODS:
    slug = slugify_method(method)
    matched_file = OUTPUT_DIR / "matched" / f"matched_{slug}_synthetic_scaled.npz"
    selected_file = OUTPUT_DIR / "selected" / f"selected_{slug}_synthetic_scaled.npz"
    assert matched_file.exists()
    assert selected_file.exists()
    matched_reload = np.load(matched_file)
    assert len(matched_reload["X_synthetic"]) == MATCHED_SYNTHETIC_COUNT
    assert set(matched_reload["y_synthetic"]) == {MATCHED_TARGET_CLASS}

print("Final augmentation acceptance checks: PASSED")
print("Next notebook: 03_ctgan_v2_full_retrain.ipynb")
